# Explainable AI for Transparent Medical Disease Classification

Run the cells **top to bottom**. The notebook builds the whole project:

| Step | What it does |
|---|---|
| 1-2 | Install packages, create folders |
| 3-6 | Write `src/data_loader.py`, `src/preprocess.py`, `src/train.py`, `src/explainers.py` |
| 7 | Download both UCI datasets and run data checks |
| 8 | Preprocess, train and compare models, save the best model per disease |
| 9 | Global SHAP plots, a local LIME explanation, and explainer timings |
| 10-11 | Write `app.py` and launch the Streamlit app |
| 12 | Check that all artifacts exist |

**Educational prototype - not for clinical diagnosis.**

In [ ]:
!pip install -q ucimlrepo shap lime streamlit scikit-learn joblib matplotlib pandas numpy

In [ ]:
from pathlib import Path

for folder in ["src", "data", "models", "results", "images"]:
    Path(folder).mkdir(exist_ok=True)
Path("src/__init__.py").touch()
print("Project folders ready.")

In [ ]:
%%writefile src/data_loader.py
"""Download the two UCI datasets and cache them as CSV files."""
import os

import pandas as pd
from ucimlrepo import fetch_ucirepo

DIABETES_ID = 529  # Early Stage Diabetes Risk Prediction
HEART_ID = 45      # Heart Disease (Cleveland)


def _fetch(dataset_id):
    """Return features + target of a UCI dataset as one DataFrame."""
    dataset = fetch_ucirepo(id=dataset_id)
    return pd.concat([dataset.data.features, dataset.data.targets], axis=1)


def load_diabetes_data():
    return _fetch(DIABETES_ID)


def load_heart_disease_data():
    return _fetch(HEART_ID)


def download_all(data_dir="data"):
    """Fetch both datasets, save them to `data_dir`, and return the DataFrames."""
    os.makedirs(data_dir, exist_ok=True)
    diabetes_df = load_diabetes_data()
    diabetes_df.to_csv(os.path.join(data_dir, "diabetes_data.csv"), index=False)
    heart_df = load_heart_disease_data()
    heart_df.to_csv(os.path.join(data_dir, "heart_disease_data.csv"), index=False)
    return diabetes_df, heart_df

In [ ]:
%%writefile src/preprocess.py
"""Preprocessing for both modules.

Both functions return (X_train, X_test, y_train, y_test) as pandas objects.
Features are NOT scaled here: scaling is done inside the Logistic Regression
pipeline (see train.py), so the saved models accept raw feature values and the
Streamlit app can pass user input straight to them. Missing values are filled
with medians computed on the training split only (no test-set leakage).
"""
import pandas as pd
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.20
RANDOM_STATE = 42

YES_NO = {"yes": 1, "no": 0}
DIABETES_TARGET_MAP = {"positive": 1, "negative": 0}


def _split_and_impute(X, y):
    """Stratified 80/20 split, then impute with training medians only."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )
    train_medians = X_train.median(numeric_only=True)
    X_train = X_train.fillna(train_medians)
    X_test = X_test.fillna(train_medians)
    return X_train, X_test, y_train, y_test


def _strict_map(series, mapping, name):
    """Map text values case-insensitively and fail loudly on unknown values."""
    mapped = series.astype(str).str.strip().str.lower().map(mapping)
    if mapped.isna().any():
        bad = sorted(series[mapped.isna()].astype(str).unique())
        raise ValueError(f"Unexpected values in '{name}': {bad}")
    return mapped.astype(int)


def preprocess_diabetes_data(df):
    df = df.copy()
    target_col = [c for c in df.columns if c.lower() in ("class", "target")][0]
    df = df.dropna(subset=[target_col])

    # Gender is dropped on purpose: in this dataset it is strongly confounded
    # with the label (a sampling artefact), so keeping it would let the model
    # and its explanations rely on a non-clinical attribute.
    gender_cols = [c for c in df.columns if c.lower() == "gender"]
    X = df.drop(columns=[target_col] + gender_cols)
    y = _strict_map(df[target_col], DIABETES_TARGET_MAP, target_col)

    # Yes/No text columns -> 1/0 (age is already numeric and is left alone)
    for col in X.columns:
        if not pd.api.types.is_numeric_dtype(X[col]):
            X[col] = _strict_map(X[col], YES_NO, col)

    return _split_and_impute(X, y)


def preprocess_heart_data(df):
    df = df.copy()
    target_col = "num" if "num" in df.columns else df.columns[-1]
    df = df.dropna(subset=[target_col])

    # num: 0 = no disease, 1-4 = disease present -> binary target
    y = (pd.to_numeric(df[target_col]) > 0).astype(int)
    X = df.drop(columns=[target_col]).apply(pd.to_numeric, errors="coerce")

    # ca / thal contain a few missing values; they are imputed after the split
    return _split_and_impute(X, y)

In [ ]:
%%writefile src/train.py
"""Train, compare and save the models for one disease module."""
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, brier_score_loss, confusion_matrix,
                             f1_score, precision_score, recall_score,
                             roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42


def build_models():
    return {
        # Trees are scale-invariant, so no scaler is needed for the forest.
        "Calibrated RF": CalibratedClassifierCV(
            RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE), cv=3
        ),
        # Scaling lives inside the pipeline, so raw inputs can be passed in.
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]),
    }


def baseline_profile(X_train):
    """Reference input: binary features = 0 (absent), other features = median.

    For the diabetes module this is "no symptoms, median age"; the predicted
    risk should be low. It is a sanity check, not a clinical profile.
    """
    row = {}
    for col in X_train.columns:
        is_binary = set(np.unique(X_train[col])) <= {0, 1}
        row[col] = 0.0 if is_binary else float(X_train[col].median())
    return pd.DataFrame([row], columns=X_train.columns)


def train_and_evaluate_all(X_train, y_train, X_test, y_test, disease_prefix="diabetes"):
    """Compare the models, pick the best one with cross-validation, and save it.

    The winner is chosen by 5-fold stratified CV ROC-AUC on the TRAINING data
    only. The held-out test set is used just for the final reported metrics.
    """
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    results, trained = [], {}

    for name, model in build_models().items():
        cv_auc = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc").mean()
        model.fit(X_train, y_train)
        probs = model.predict_proba(X_test)[:, 1]
        preds = (probs >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, preds, labels=[0, 1]).ravel()
        results.append({
            "Model": name,
            "CV ROC-AUC": cv_auc,
            "Test ROC-AUC": roc_auc_score(y_test, probs),
            "Accuracy": accuracy_score(y_test, preds),
            "Precision": precision_score(y_test, preds, zero_division=0),
            "Recall": recall_score(y_test, preds, zero_division=0),
            "F1": f1_score(y_test, preds, zero_division=0),
            "Specificity": tn / (tn + fp) if (tn + fp) else float("nan"),
            "Brier": brier_score_loss(y_test, probs),
        })
        trained[name] = model

    res_df = pd.DataFrame(results)
    best_name = res_df.sort_values("CV ROC-AUC", ascending=False).iloc[0]["Model"]
    best_model = trained[best_name]

    base_risk = best_model.predict_proba(baseline_profile(X_train))[0][1]
    print(f"[{disease_prefix.upper()}] Best model (by CV ROC-AUC): {best_name} | "
          f"Baseline-profile risk: {base_risk:.2%}")

    os.makedirs("models", exist_ok=True)
    os.makedirs("results", exist_ok=True)
    joblib.dump(best_model, f"models/{disease_prefix}_best_model.pkl")
    joblib.dump(list(X_train.columns), f"models/{disease_prefix}_features.pkl")
    res_df.to_csv(f"results/{disease_prefix}_metrics.csv", index=False)
    return res_df, trained

In [ ]:
%%writefile src/explainers.py
"""SHAP and LIME helpers shared by the notebook and the Streamlit app.

Both explainers work on the model's positive-class probability and are
model-agnostic, so they work for the calibrated forest and the logistic
regression pipeline alike. Inputs are always DataFrames of RAW (unscaled)
features in the same column order used for training.
"""
import time

import numpy as np
import pandas as pd
import shap
from lime.lime_tabular import LimeTabularExplainer

SEED = 42


def make_predict_fn(model, feature_names):
    """f(array) -> P(positive); restores column names to avoid sklearn warnings."""
    names = list(feature_names)

    def predict_positive(data):
        return model.predict_proba(pd.DataFrame(np.asarray(data), columns=names))[:, 1]

    return predict_positive


def make_predict_proba_fn(model, feature_names):
    """f(array) -> [P(negative), P(positive)] for LIME."""
    names = list(feature_names)

    def predict_proba(data):
        return model.predict_proba(pd.DataFrame(np.asarray(data), columns=names))

    return predict_proba


def build_shap_explainer(model, X_train, max_background=100):
    names = list(X_train.columns)
    background = shap.sample(X_train, min(max_background, len(X_train)), random_state=SEED)
    masker = shap.maskers.Independent(background, max_samples=len(background))
    return shap.Explainer(make_predict_fn(model, names), masker, feature_names=names)


def generate_shap_explanations(model, X_train, X_test, feature_names=None):
    """Global SHAP values for X_test.

    Returns (explainer, shap_values, values_array, seconds).
    `shap_values` is a shap.Explanation; `values_array` has shape (n, features)
    and is what shap.summary_plot expects.
    """
    names = list(feature_names) if feature_names is not None else list(X_train.columns)
    X_tr = pd.DataFrame(X_train, columns=names)
    X_te = pd.DataFrame(X_test, columns=names)

    start = time.time()
    explainer = build_shap_explainer(model, X_tr)
    shap_values = explainer(X_te)
    elapsed = time.time() - start
    return explainer, shap_values, shap_values.values, elapsed


def build_lime_explainer(X_train, class_names=("Negative", "Positive")):
    names = list(X_train.columns)
    # Binary 0/1 columns are treated as categorical so LIME does not bin them.
    categorical = [i for i, c in enumerate(names) if set(np.unique(X_train[c])) <= {0, 1}]
    return LimeTabularExplainer(
        training_data=np.asarray(X_train),
        feature_names=names,
        class_names=list(class_names),
        categorical_features=categorical,
        mode="classification",
        random_state=SEED,
    )


def generate_lime_explanation(model, X_train, instance, feature_names=None,
                              class_names=("Negative", "Positive"), num_features=10):
    """LIME explanation for one patient. Returns (explanation, seconds)."""
    names = list(feature_names) if feature_names is not None else list(X_train.columns)
    X_tr = pd.DataFrame(X_train, columns=names)

    start = time.time()
    explainer = build_lime_explainer(X_tr, class_names)
    exp = explainer.explain_instance(
        data_row=np.asarray(instance, dtype=float).ravel(),
        predict_fn=make_predict_proba_fn(model, names),
        num_features=num_features,
    )
    return exp, time.time() - start

In [ ]:
import os, sys
import pandas as pd

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from src.data_loader import download_all

print("Downloading Early Stage Diabetes Risk Prediction (ID 529) and Cleveland Heart Disease (ID 45)...")
diabetes_df, heart_df = download_all("data")

for name, df, target in [("Diabetes", diabetes_df, "class"), ("Heart disease", heart_df, "num")]:
    print(f"\n--- {name}: shape {df.shape} ---")
    print("Missing values per column:", df.isna().sum()[df.isna().sum() > 0].to_dict() or "none")
    print("Duplicate rows:", int(df.duplicated().sum()))
    print("Target distribution:", df[target].value_counts().to_dict())

In [ ]:
import joblib
import pandas as pd
from IPython.display import display

from src.preprocess import preprocess_diabetes_data, preprocess_heart_data
from src.train import train_and_evaluate_all

diabetes_df = pd.read_csv("data/diabetes_data.csv")
heart_df = pd.read_csv("data/heart_disease_data.csv")


def test_rows_seen_in_training(X_train, X_test):
    """Count test rows that are exact copies of a training row (possible optimism)."""
    train_rows = set(map(tuple, X_train.round(6).values))
    return sum(tuple(r) in train_rows for r in X_test.round(6).values)


print("=== Diabetes pipeline ===")
X_tr_d, X_te_d, y_tr_d, y_te_d = preprocess_diabetes_data(diabetes_df)
print(f"Train {X_tr_d.shape}, Test {X_te_d.shape}")
print(f"Test rows identical to a training row: {test_rows_seen_in_training(X_tr_d, X_te_d)} of {len(X_te_d)}")
d_metrics, d_models = train_and_evaluate_all(X_tr_d, y_tr_d, X_te_d, y_te_d, "diabetes")
display(d_metrics.round(4).style.set_caption("Diabetes - model comparison (test set)"))

print("\n=== Heart disease pipeline ===")
X_tr_h, X_te_h, y_tr_h, y_te_h = preprocess_heart_data(heart_df)
print(f"Train {X_tr_h.shape}, Test {X_te_h.shape}")
print(f"Test rows identical to a training row: {test_rows_seen_in_training(X_tr_h, X_te_h)} of {len(X_te_h)}")
h_metrics, h_models = train_and_evaluate_all(X_tr_h, y_tr_h, X_te_h, y_te_h, "heart")
display(h_metrics.round(4).style.set_caption("Heart disease - model comparison (test set)"))

In [ ]:
import time
import matplotlib.pyplot as plt
import shap

from src.explainers import generate_shap_explanations, generate_lime_explanation

best_diabetes_model = joblib.load("models/diabetes_best_model.pkl")
best_heart_model = joblib.load("models/heart_best_model.pkl")

# ---- Global SHAP: diabetes
print("=== Global SHAP: Diabetes ===")
d_explainer, d_shap_vals, d_s_array, d_shap_time = generate_shap_explanations(
    best_diabetes_model, X_tr_d, X_te_d)
shap.summary_plot(d_s_array, X_te_d, show=False)
plt.title("Diabetes model - global SHAP feature importance")
plt.savefig("images/shap_diabetes.png", dpi=150, bbox_inches="tight")
plt.show()

# ---- Global SHAP: heart disease
print("\n=== Global SHAP: Heart disease ===")
h_explainer, h_shap_vals, h_s_array, h_shap_time = generate_shap_explanations(
    best_heart_model, X_tr_h, X_te_h)
shap.summary_plot(h_s_array, X_te_h, show=False)
plt.title("Heart disease model - global SHAP feature importance")
plt.savefig("images/shap_heart.png", dpi=150, bbox_inches="tight")
plt.show()

# ---- Local explanations for the first test patient (diabetes)
patient = X_te_d.iloc[[0]]
start = time.time()
d_explainer(patient)
shap_single_time = time.time() - start

d_lime_exp, d_lime_time = generate_lime_explanation(
    best_diabetes_model, X_tr_d, patient, class_names=["No Diabetes", "Diabetes"])

print("\n=== Explainer runtime comparison ===")
print(f"SHAP, whole test set ({len(X_te_d)} patients): {d_shap_time:.2f}s")
print(f"SHAP, single patient:                  {shap_single_time:.2f}s")
print(f"LIME, single patient:                  {d_lime_time:.2f}s")

d_lime_exp.as_pyplot_figure()
plt.tight_layout()
plt.savefig("images/lime_diabetes_patient0.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
%%writefile app.py
import os
import sys
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import streamlit as st

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from src.explainers import (build_lime_explainer, build_shap_explainer,  # noqa: E402
                            make_predict_proba_fn)
from src.preprocess import preprocess_diabetes_data, preprocess_heart_data  # noqa: E402

st.set_page_config(page_title="Explainable AI for Transparent Medical Disease Classification",
                   layout="wide")

CONDITIONS = {
    "Diabetes": {
        "prefix": "diabetes",
        "csv": "data/diabetes_data.csv",
        "preprocess": preprocess_diabetes_data,
    },
    "Heart Disease": {
        "prefix": "heart",
        "csv": "data/heart_disease_data.csv",
        "preprocess": preprocess_heart_data,
    },
}

YES_NO = {"No": 0, "Yes": 1}

# Cleveland heart-disease fields: label, and value mapping for categorical ones.
# Codes follow the UCI documentation of the dataset.
HEART_FIELDS = {
    "age": ("Age (years)", None),
    "sex": ("Sex", {"Female": 0, "Male": 1}),
    "cp": ("Chest pain type", {"Typical angina": 1, "Atypical angina": 2,
                               "Non-anginal pain": 3, "Asymptomatic": 4}),
    "trestbps": ("Resting blood pressure (mm Hg)", None),
    "chol": ("Serum cholesterol (mg/dl)", None),
    "fbs": ("Fasting blood sugar > 120 mg/dl", YES_NO),
    "restecg": ("Resting ECG result", {"Normal": 0, "ST-T wave abnormality": 1,
                                       "Left ventricular hypertrophy": 2}),
    "thalach": ("Maximum heart rate achieved (bpm)", None),
    "exang": ("Exercise-induced angina", YES_NO),
    "oldpeak": ("ST depression induced by exercise", None),
    "slope": ("Slope of peak exercise ST segment", {"Upsloping": 1, "Flat": 2,
                                                    "Downsloping": 3}),
    "ca": ("Major vessels coloured by fluoroscopy (0-3)", {"0": 0, "1": 1, "2": 2, "3": 3}),
    "thal": ("Thalassemia", {"Normal": 3, "Fixed defect": 6, "Reversible defect": 7}),
}


@st.cache_resource
def load_resources(condition):
    """Load the saved model plus the training data used to build the explainers."""
    cfg = CONDITIONS[condition]
    model = joblib.load(f"models/{cfg['prefix']}_best_model.pkl")
    feature_names = joblib.load(f"models/{cfg['prefix']}_features.pkl")
    raw_df = pd.read_csv(cfg["csv"])
    # Re-run the exact training preprocessing so explainers see the same data
    X_train = cfg["preprocess"](raw_df)[0][feature_names]
    return {
        "model": model,
        "features": feature_names,
        "X_train": X_train,
        "shap": build_shap_explainer(model, X_train),
        "lime": build_lime_explainer(X_train, class_names=("Negative", "Positive")),
    }


def pretty(name):
    return name.replace("_", " ").capitalize()


def collect_inputs(condition, feature_names, X_train):
    """Render sidebar widgets and return one row of RAW feature values."""
    values = {}
    for col in feature_names:
        median = float(X_train[col].median())
        if condition == "Diabetes":
            if col == "age":
                values[col] = st.sidebar.number_input("Age (years)", 1, 120, int(round(median)))
            else:
                values[col] = YES_NO[st.sidebar.selectbox(pretty(col), list(YES_NO))]
        else:
            label, options = HEART_FIELDS.get(col, (pretty(col), None))
            if options is None:  # numeric input
                step = 0.1 if col == "oldpeak" else 1.0
                values[col] = st.sidebar.number_input(
                    label, min_value=0.0, value=median, step=step)
            else:
                values[col] = options[st.sidebar.selectbox(label, list(options))]
    return pd.DataFrame([values], columns=feature_names).astype(float)


st.title("Explainable AI for Transparent Medical Disease Classification")
st.warning("Educational prototype only - not a diagnostic tool and not medical advice. "
           "The models were trained on small public datasets and describe EXISTING "
           "disease status in those datasets, not whether a person will develop a disease.")

condition = st.sidebar.selectbox("Condition", list(CONDITIONS))
res = load_resources(condition)
model, feature_names, X_train = res["model"], res["features"], res["X_train"]

st.sidebar.header("Patient input")
input_df = collect_inputs(condition, feature_names, X_train)

prob = float(model.predict_proba(input_df)[0][1])
st.header(f"Classification result: {condition}")
st.metric("Estimated probability of a positive classification", f"{prob:.2%}")
if prob >= 0.5:
    st.error("Classified as POSITIVE (disease pattern present in the model's training data)")
else:
    st.success("Classified as NEGATIVE")

st.header("Why? Feature contributions")
tab_shap, tab_lime = st.tabs(["SHAP", "LIME"])

with tab_shap:
    start = time.time()
    sv = res["shap"](input_df)
    shap_secs = time.time() - start
    shap.plots.waterfall(sv[0], show=False)
    st.pyplot(plt.gcf())
    plt.close("all")
    st.caption(f"Red bars push the probability up, blue bars push it down. "
               f"Computed in {shap_secs:.2f}s.")

with tab_lime:
    start = time.time()
    exp = res["lime"].explain_instance(
        input_df.values[0], make_predict_proba_fn(model, feature_names), num_features=8)
    lime_secs = time.time() - start
    st.pyplot(exp.as_pyplot_figure())
    plt.close("all")
    st.caption(f"Green bars support a positive classification, red bars a negative one. "
               f"Computed in {lime_secs:.2f}s. LIME is a local approximation and can vary "
               f"slightly between runs.")

st.caption("Explanations show what the model relied on; they do not prove medical "
           "cause or clinical correctness.")

In [ ]:
import subprocess
import time

# Restart any running Streamlit server so the new app.py is served
subprocess.run(["pkill", "-f", "streamlit"], check=False)
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false",
])
time.sleep(6)  # give the server time to start

try:  # Google Colab: show the app inside the notebook and print a public link
    from google.colab import output
    from google.colab.output import eval_js
    output.serve_kernel_port_as_iframe(8501, height="1000")
    print("Open in a new tab:", eval_js("google.colab.kernel.proxyPort(8501)"))
except ImportError:  # local Jupyter
    print("Running locally - open http://localhost:8501 in your browser.")

In [ ]:
import os

required_files = [
    "data/diabetes_data.csv", "data/heart_disease_data.csv",
    "models/diabetes_best_model.pkl", "models/diabetes_features.pkl",
    "models/heart_best_model.pkl", "models/heart_features.pkl",
    "results/diabetes_metrics.csv", "results/heart_metrics.csv", "app.py",
]
print("Project artifact audit:")
for f in required_files:
    print(f"- {f}: {'EXISTS' if os.path.exists(f) else 'MISSING'}")

### Notes
- **No scaler artifact:** scaling is part of the Logistic Regression pipeline inside the saved model, so the app passes raw feature values straight to `model.predict_proba`.
- **Model selection** uses 5-fold stratified CV ROC-AUC on the training split; the test set is only used for the reported metrics.
- **Gender** is dropped from the diabetes module (it is confounded with the label in this dataset).
- **Run the Streamlit app locally** with `streamlit run app.py` after executing the notebook once (or after running the same steps as scripts).